# Vox Machina — Getting Started

This notebook walks through the core capabilities of the Vox Machina library:
loading audio, detecting pitch, applying auto-tune, running a vocoder, and adding effects.

In [ ]:
# Install the library (run once)
# !pip install -e /path/to/VoxMachinae

## 1. Loading Audio

Vox Machina uses `AudioBuffer` as its core data structure — a NumPy array with metadata.

In [ ]:
from voxmachinae.core.audio_io import load_audio, save_audio
import numpy as np

# Load a sample audio file
# audio = load_audio("your_vocal.wav")

# For this demo, create a synthetic vocal-like signal
from voxmachinae.core.audio_io import AudioBuffer

sr = 44100
duration = 2.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Simulate a voice with vibrato (A3 = 220 Hz)
vibrato = 5.0 * np.sin(2 * np.pi * 5.5 * t)  # 5.5 Hz vibrato
signal = 0.5 * np.sin(2 * np.pi * (220 + vibrato) * t)

# Add some harmonics for richness
signal += 0.25 * np.sin(2 * np.pi * (440 + vibrato * 2) * t)
signal += 0.12 * np.sin(2 * np.pi * (660 + vibrato * 3) * t)

audio = AudioBuffer(samples=signal.astype(np.float32), sample_rate=sr)
print(f"Audio: {audio.duration:.2f}s, {audio.sample_rate} Hz, {audio.samples.shape}")

## 2. Pitch Detection

Detect the fundamental frequency over time using pYIN (default) or CREPE.

In [ ]:
from voxmachinae.core.pitch import detect_pitch

track = detect_pitch(audio, method="pyin")

print(f"Detected {len(track.times)} pitch frames")
print(f"Voiced frames: {track.voiced.sum()} / {len(track.voiced)}")

voiced_freqs = track.frequencies[track.voiced]
if len(voiced_freqs) > 0:
    print(f"Frequency range: {voiced_freqs.min():.1f} - {voiced_freqs.max():.1f} Hz")
    print(f"Mean frequency: {voiced_freqs.mean():.1f} Hz")

## 3. Auto-Tune

Apply pitch correction with different styles — from natural to robotic.

In [ ]:
from voxmachinae.core.autotune import AutoTune, AutoTuneParams

# Natural correction
natural_params = AutoTuneParams(
    key="A",
    scale="minor",
    retune_speed=60,
    humanize=40,
)
tuner = AutoTune(natural_params)
natural_result = tuner.process(audio)
print(f"Natural auto-tune: {natural_result.audio.duration:.2f}s")

# T-Pain / robotic style
robotic_params = AutoTuneParams(
    key="A",
    scale="minor",
    retune_speed=0,   # instant snap
    humanize=0,       # no variation preserved
)
tuner_robotic = AutoTune(robotic_params)
robotic_result = tuner_robotic.process(audio)
print(f"Robotic auto-tune: {robotic_result.audio.duration:.2f}s")

## 4. Vocoder

Use a channel vocoder to impose voice characteristics onto a synthesizer carrier.

In [ ]:
from voxmachinae.core.vocoder import ChannelVocoder, ChannelVocoderParams
from voxmachinae.synthesis.oscillators import generate_waveform

# Create a sawtooth carrier
carrier = generate_waveform(
    "sawtooth",
    frequency=110.0,
    duration=audio.duration,
    sample_rate=audio.sample_rate,
)

# 16-band channel vocoder
params = ChannelVocoderParams(num_bands=16, carrier_type="sawtooth")
vocoder = ChannelVocoder(params)
result = vocoder.process(audio, carrier)
print(f"Vocoded output: {result.duration:.2f}s")

## 5. Effects

Apply reverb, delay, and formant shifting.

In [ ]:
from voxmachinae.core.effects import (
    apply_reverb, ReverbParams,
    apply_delay, DelayParams,
    apply_formant_shift, FormantShiftParams,
)

# Large hall reverb
reverbed = apply_reverb(audio, ReverbParams(room_size=0.85, damping=0.4, wet=0.35))
print(f"Reverbed: {reverbed.duration:.2f}s")

# Rhythmic delay
delayed = apply_delay(audio, DelayParams(delay_time=0.25, feedback=0.5, mix=0.3))
print(f"Delayed: {delayed.duration:.2f}s")

# Formant shift (higher voice character)
shifted = apply_formant_shift(audio, FormantShiftParams(shift_semitones=4.0))
print(f"Formant shifted: {shifted.duration:.2f}s")

## 6. Using Presets

Load pre-configured parameter sets for classic sounds.

In [ ]:
from voxmachinae.presets.autotune_presets import AUTOTUNE_PRESETS
from voxmachinae.presets.vocoder_presets import CHANNEL_VOCODER_PRESETS

print("Available auto-tune presets:")
for name in AUTOTUNE_PRESETS:
    print(f"  - {name}")

print("\nAvailable vocoder presets:")
for name in CHANNEL_VOCODER_PRESETS:
    print(f"  - {name}")

## 7. Musical Scales

Work with scales, key detection, and note frequencies.

In [ ]:
from voxmachinae.core.scales import get_scale_frequencies, SCALES

# List available scales
print("Available scales:")
for scale_name in sorted(SCALES.keys()):
    intervals = SCALES[scale_name]
    print(f"  {scale_name}: {intervals}")

# Get frequencies for C major
freqs = get_scale_frequencies("C", "major")
print(f"\nC major frequencies (first octave): {[f'{f:.1f}' for f in freqs[:8]]}")

## Next Steps

- Try the **web app** for an interactive interface: `cd webapp && uvicorn backend.main:app`
- Explore the [API Reference](https://voxmachinae.dev/api/) for full parameter documentation
- Check the [DSP Glossary](https://voxmachinae.dev/reference/glossary/) for concept explanations